In [1]:
import os
import pandas as pd
import numpy as np

metrics_dir = "./augmented_datasets_metrics"


In [37]:
headers = ["model", "method", "dataset", "task", "augmentation_rate", "tsed_score"]
all_data = []

for root, _, files in os.walk(metrics_dir):
    if "paraphrase" in root:
        continue

    for file in files:
        if not file.endswith(".csv"):
            continue

        file_path = os.path.join(root, file)
        df = pd.read_csv(file_path)

        if df.empty or "augmentation_rate" not in df.columns:
            continue

        df["augmentation_rate"] = df["augmentation_rate"].astype(float)
        # skip all 0.05 steps
        df = df[(df["augmentation_rate"] * 100).astype(int) % 10 == 0]
        df.sort_values("augmentation_rate", inplace=True)

        valid_rates = df["augmentation_rate"].value_counts()
        valid_rates = valid_rates[valid_rates == 5].index
        df = df[df["augmentation_rate"].isin(valid_rates)]
        df = df[df['tsed_score'].notna()]

        if df.empty:
            continue

        for (i, row) in df.iterrows():
            all_data.append([row["model"], row["method"], row["dataset"], row["task"], row["augmentation_rate"], row["tsed_score"]])

df = pd.DataFrame(all_data, columns=headers)

Use Friedman test to prove that increase of sensitivity with augmentation rate is not random. Test reference: https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.friedmanchisquare.html#scipy.stats.friedmanchisquare

In [38]:
from scipy.stats import friedmanchisquare

aug_rates = df["augmentation_rate"].unique().tolist()

for method in df["method"].unique().tolist():
    print(f"Method: {method}")
    aug_rates_stats = {ar:[] for ar in aug_rates}
    for aug_rate in aug_rates_stats:
        for model in df["model"].unique().tolist():
            sub_df = df[(df["method"] == method) & (df["model"] == model) & (df["augmentation_rate"] == aug_rate)]
            aug_rates_stats[aug_rate].append(sub_df["tsed_score"].mean())
    
    test_result = friedmanchisquare(*aug_rates_stats.values())
    print(f"p value: {test_result.pvalue}")



Method: synonym
p value: 0.0003800125923898571
Method: keyboard
p value: 2.7823285276458548e-05


Now use Kruskal-Wallis test to validate that results between datasets significantly differ.

In [41]:
from scipy.stats import kruskal

for method in df["method"].unique().tolist():
    print(f"Method: {method}")
    dataset_stats = {d:[] for d in df["dataset"].unique().tolist()}
    for dataset in dataset_stats:
        sub_df = df[(df["method"] == method) & (df["dataset"] == dataset)]
        dataset_stats[dataset] = sub_df.groupby("augmentation_rate")[["tsed_score"]].mean()["tsed_score"].tolist()


    test_result = kruskal(*dataset_stats.values())
    print(f"p value: {test_result.pvalue}")



Method: synonym
p value: 2.210969386177391e-05
Method: keyboard
p value: 0.015419917140115712


## Paraphrasing separately

In [33]:
headers = ["model", "dataset", "augmentation_rate", "tsed_score"]
all_data = []

for root, _, files in os.walk(metrics_dir):
    if "paraphrase" not in root:
        continue

    for file in files:
        if not file.endswith(".csv"):
            continue

        file_path = os.path.join(root, file)
        df = pd.read_csv(file_path)

        if df.empty or "augmentation_rate" not in df.columns:
            continue

        df["augmentation_rate"] = df["augmentation_rate"].astype(float)
        df.sort_values("augmentation_rate", inplace=True)

        valid_rates = df["augmentation_rate"].value_counts()
        valid_rates = valid_rates[valid_rates == 5].index
        df = df[df["augmentation_rate"].isin(valid_rates)]
        df = df[df['tsed_score'].notna()]

        if df.empty:
            continue

        for (i, row) in df.iterrows():
            all_data.append([row["model"], row["dataset"], row["augmentation_rate"], row["tsed_score"]])

para_df = pd.DataFrame(all_data, columns=headers)

Again, we perform Friedman test

In [ ]:
aug_rates = para_df["augmentation_rate"].unique().tolist()

aug_rates_stats = {ar:[] for ar in aug_rates}
for aug_rate in aug_rates_stats:
    for model in para_df["model"].unique().tolist():
        sub_df = para_df[(para_df["model"] == model) & (para_df["augmentation_rate"] == aug_rate)]
        aug_rates_stats[aug_rate].append(sub_df["tsed_score"].mean())

test_result = friedmanchisquare(*aug_rates_stats.values())
print(f"p value: {test_result.pvalue}")

p value: 0.011197246715144049
